# Face Attribute Editing - Two-Model Pipeline

This notebook is a lightweight entry point for the final repository pipeline.
It compares the SDXL base 1.0 LoRA baseline (`sdxl`) against the Stable Diffusion 1.5 LoRA comparison model (`sd15`) on three mask-guided face attribute editing tasks:

- `add_eyeglasses`
- `make_smiling`
- `make_older`

The implementation lives in `src/`; this notebook only wires the main steps together.

## 1. Environment Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

## 2. Configuration

In [ ]:
from src.config import MODELS, PATHS, SEED, TASKS, seed_everything

seed_everything(SEED)
print("Tasks:", TASKS)
print("Models:", list(MODELS))
for key, value in PATHS.items():
    print(f"{key}: {value}")

## 3. Optional Data Download

In [ ]:
from src.data.download import download_if_needed, print_raw_data_status

RUN_DOWNLOAD = False
print_raw_data_status()
if RUN_DOWNLOAD:
    download_if_needed()

## 4. Preprocessing And Manifests

In [ ]:
from src.data.manifest import build_manifests
from src.data.preprocess import run_preprocessing

RUN_PREPROCESSING = False
SMOKE_TEST = True
max_images = 200 if SMOKE_TEST else 2000

if RUN_PREPROCESSING:
    records = run_preprocessing(max_images=max_images, force=False)
    splits = build_manifests(records, seed=SEED, force=False)
else:
    print("Skipping preprocessing. Existing manifests are expected under:", PATHS["manifest_dir"])

## 5. Train Attribute Classifier

In [ ]:
from src.models.classifier import train_classifier

RUN_CLASSIFIER_TRAINING = False
if RUN_CLASSIFIER_TRAINING:
    classifier_ckpt = train_classifier(smoke=SMOKE_TEST, force=False)
else:
    classifier_ckpt = PATHS["run_dir"] / "checkpoints" / "attr_classifier" / "best.pt"
print("Classifier checkpoint:", classifier_ckpt, "exists=", classifier_ckpt.exists())

## 6. Train LoRA Adapters

In [ ]:
from src.models.lora_training import build_diffusers_imagefolder, run_lora_training

RUN_LORA_TRAINING = False
MODELS_TO_TRAIN = ["sd15", "sdxl"]

if RUN_LORA_TRAINING:
    split_name = "train_smoke" if SMOKE_TEST else "train"
    for model_id in MODELS_TO_TRAIN:
        data_dir = build_diffusers_imagefolder(model_id, split_name=split_name, force=False)
        run_lora_training(model_id, data_dir, smoke=SMOKE_TEST, force=False)
else:
    print("Skipping LoRA training. Expected adapters:")
    for model_id in MODELS_TO_TRAIN:
        print(" ", PATHS["run_dir"] / "loras" / model_id)

## 7. Batch Inference

In [ ]:
from src.inference.batch_edit import batch_edit_model

RUN_INFERENCE = False
MODELS_TO_RUN = ["sd15", "sdxl"]
split_name = "test_smoke" if SMOKE_TEST else "test"
samples_per_task = 2 if SMOKE_TEST else 50

if RUN_INFERENCE:
    for model_id in MODELS_TO_RUN:
        edited_meta = batch_edit_model(
            model_id=model_id,
            split_name=split_name,
            samples_per_task=samples_per_task,
        )
        print(model_id, "edited images:", len(edited_meta))
else:
    print("Skipping inference. Existing edited outputs are expected under:", PATHS["run_dir"] / "edited")

## 8. Evaluation

In [ ]:
from src.evaluation.evaluate import evaluate_edits

RUN_EVALUATION = False
if RUN_EVALUATION:
    results_long, results_summary = evaluate_edits()
    display(results_summary)
else:
    results_long = None
    results_summary = None
    print("Skipping evaluation.")

## 9. Benchmark Figures And Report

In [ ]:
import pandas as pd
from src.evaluation.benchmark import run_benchmark
from src.visualization.qualitative_grid import generate_qualitative_grid
from src.visualization.report import generate_report

RUN_REPORTING = False
if RUN_REPORTING:
    summary_path = PATHS["run_dir"] / "metrics" / "results_summary.csv"
    long_path = PATHS["run_dir"] / "metrics" / "results_long.csv"
    results_csvs = {model_id: summary_path for model_id in MODELS} if summary_path.exists() else None
    benchmark = run_benchmark(results_csvs=results_csvs)
    if long_path.exists():
        results_long = pd.read_csv(long_path)
        qual_paths = generate_qualitative_grid(
            results_long,
            PATHS["run_dir"] / "reports" / "qualitative",
        )
    else:
        qual_paths = []
    summary = pd.read_csv(summary_path) if summary_path.exists() else None
    report = generate_report(summary, qual_paths, smoke=SMOKE_TEST)
    print("Report:", report)
else:
    print("Skipping reporting.")